In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

from joblib import Parallel, delayed
from matplotlib.lines import Line2D
from pandas.tseries.offsets import MonthEnd
from scipy.optimize import minimize
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error as root_mse
from tqdm import tqdm

os.makedirs("output", exist_ok=True)

# Données

In [ ]:
model_data = pd.read_parquet("data/model_data.parquet")
model_data_no_journal = pd.read_parquet("data/model_data_njl.parquet")
nuances_order = ["Far right", "Right", "Center", "Left", "Far left", "Other"]

In [ ]:
presi_dates = [
    "26/04/1981", "10/05/1981",
    "24/04/1988", "08/05/1988",
    "23/04/1995", "07/05/1995",
    "21/04/2002", "05/05/2002",
    "22/04/2007", "06/05/2007",
    "22/04/2012", "06/05/2012",
    "21/04/2017", "07/05/2017",
    "10/04/2022", "24/04/2022"]
presi_months = pd.to_datetime(presi_dates, dayfirst=True).to_period('M').drop_duplicates()

legi_dates = [
    "14/06/1981", "21/06/1981",
    "16/03/1986",
    "05/06/1988", "11/06/1988",
    "21/03/1993", "28/03/1993",
    "25/05/1997", "01/06/1997",
    "09/06/2002", "16/06/2002",
    "10/06/2007", "17/06/2007",
    "10/06/2012", "17/06/2012",
    "11/06/2017", "18/06/2017",
    "12/06/2022", "19/06/2022",
    "29/06/2024", "06/07/2024"]
legi_months = pd.to_datetime(legi_dates, dayfirst=True).to_period('M').drop_duplicates()

europ_dates = [
    "17/06/1984",
    "18/06/1989",
    "12/06/1994",
    "13/06/1999",
    "13/06/2004",
    "07/06/2009",
    "25/05/2014",
    "26/05/2019",
    "09/06/2024"]
europ_months = pd.to_datetime(europ_dates, dayfirst=True).to_period('M').drop_duplicates()

main_elec_dates = presi_dates + legi_dates + europ_dates
main_elec_months = pd.to_datetime(main_elec_dates, dayfirst=True).to_period('M').drop_duplicates().sort_values()

def add_shaded_periods(ax_list, periods, color, alpha):
    start_period = None
    for i, period in enumerate(periods):
        if start_period is None:
            start_period = period
        is_last = (i == len(periods) - 1)
        is_gap = (not is_last and periods[i + 1] != period + 1)
        if is_last or is_gap:
            end_period = period
            start = start_period.to_timestamp()
            end = (end_period + MonthEnd(1)).to_timestamp()
            for ax in ax_list:
                ax.axvspan(start, end, color=color, alpha=alpha)
            start_period = None

In [ ]:
outcome = "quotes_share"

# 1. Méthode

## Modèle

L'organisation du système politique français fait que la couverture médiatique de la vie politique à l'échelle nationale est structurée par le cycle des élections présidentielles. On décompose celui-ci en trois périodes répondant à des logiques différentes, c'est-à-dire où la norme de représentativité s'incarne différemment :
- La période inter-électorale, où la couverture médiatique des nuances politiques dépend de leur importance électorale et institutionnelle
- La période pré-électorale, correspondant à la campagne électorale jusqu'au premier tour de l'élection présidentielle, où les sondages prennent un poids croissant
- La période post-électorale, recouvrant le second tour de l'élection présidentielle et les élections législatives

Indiçons par $i$ les nuances politiques ($n$ au total) et $t$ les périodes, puis notons $Y_{it}$ la proportion des citations reçues par la nuance $i$ à la période $t$ et $\hat{Y_{it}}$ qu'elle devrait normalement recevoir. Les modèles sont formulés de manière à respecter les contraintes de compositionnalité pour des coefficients compris entre 0 et 1.

**Période inter-électorale**

Supposons qu'à chaque période :
- Toutes les nuances reçoivent une fraction incompressible des citations, c'est-à-dire qu'elles bénéficient d'une couverture médiatique minimale même lorsque leur importance électorale et institutionnelle est nul. Cette fraction est notée $\alpha$. L'hypothèse $\alpha > 0$ est plausible dans la mesure où les nuances recouvrent plusieurs partis et où l'unité d'observation est le mois. Elle permet aussi d'introduire une constante dans le modèle, qui s'avère souvent nécessaire pour centrer les résidus.
- La nuance au gouvernement reçoit une fraction fixe des citations, notée $\theta$, reflétant la couverture médiatique de l'action gouvernementale. Notons $G_{it}$ l'indicatrice valant 1 si le Premier ministre appartient à la nuance politique $i$ durant la période $t$. Lorsqu'une autre nuance politique est représentée au gouvernement, $G_{it} = \frac{1}{2}$ pour chacune, de sorte que $\sum_i G_{it}$ vaut toujours 1.
- Les citations restantes sont réparties entre les nuances politiques en fonction de leur poids dans la vie politique, estimé par la moyenne pondérée de la proportion des sièges qu'elles détiennent à l'Assemblée nationale (notée ${AN}_{it}$ et pondérée par $\beta$) et du score qu'elles ont obtenu au premier tour de la dernière élection présidentielle (notée $P_{it}$ et pondérée par $\gamma$).
  - Les sièges à l'AN semblent le meilleur proxy de l'importance institutionnelle des nuances, leur participation au gouvernement étant déjà connue. Nous utilisons la nuance des groupes parlementaires plutôt que des élus.
  - Le premier tour des élections présidentielles reflète le mieux les préférences politiques des Français, car les candidats sont identiques sur l'ensemble du territoire et appartiennent à toutes les nuances politiques, mais aussi car la participation est très élevée. Nous retenons la nuance des candidats.

$$\hat{Y}_{it}^{inter} = \alpha + (1 - n\alpha - \theta)(\beta {AN}_{it} + \gamma P_{it}) + \theta G_{it}$$

**Période pré-électorale (novembre-avril)**

Le modèle précédent continue à fournir un niveau de référence, mais la couverture médiatique de la vie politique intègre désormais les résultats des sondages, notés $S_{it}$, qui peuvent conduire à s'en éloigner. L'importance des sondages devrait s'accroître à proximité du scrutin, dont la distance est reflétée par $e$ (de $e=0$ pour avril à $e=-5$ pour novembre). Cela s'écrit :

$$\hat{Y}_{it}^{pre} = \alpha + \delta_t^e \bigr[(1 - n\alpha - \theta)(\beta {AN}_{it} + \gamma P_{it}) + \theta G_{it}\bigr] + (1 - \delta_t^e) S_{it}$$

Ce modèle aura mécaniquement pour effet de réduire la proportion des citations attribuées aux nuances dominantes, même lorsque les sondages les créditent d'un meilleur score que durant la précédente élection présidentielle. Les modalités d'organisation des élections législatives conduisent en effet à amplifier leur domination, de manière à générer des majorités stables. Les partis dominants détiennent toujours une proportion de députés supérieure à leur score au premier tour de l'élection présidentielle, même en cas de dissolution. Que la pondération de leur importance institutionnelle soit diminuée par $\delta_t^e$ en période pré-électorale signifie que les journaux doivent accroître leur couverture des nuances politiques marginales, en considérant leur importance politique et institutionnelle *potentielle* et afin que la campagne électorale soit sincère.

**Période post-électorale (mai-juin)**

Il s'agit ici de traiter le décalage entre les élections présidentielles et législatives, qui engendre des discontinuités dans les prévisions si l'on actualise le niveau de référence immédiatement après le premier tour des présidentielles (puisqu'il dépendra pendant 2 mois des résultats des précédentes législatives, qui ne reflètent plus aussi bien l'importance électorale et institutionnelle des nuances). Considérons que la couverture médiatique de la vie politique est dominée par les résultats du premier tour des présidentielles :
- Les 2 nuances arrivées en tête, qui vont s'affronter au second tour, reçoivent des fractions fixes des citations, notées $\lambda_1$ et $\lambda_2$
- Les citations restantes sont réparties entre l'ensemble des nuances en fonction de leur score.

En notant $P_{it}^{r}$ l'indicatrice valant 1 si le parti $i$ est arrivé au rang $r$ au premier tour, cela s'écrit :

$$\hat{Y}_{it}^{post} = \alpha + (1 - n\alpha - \lambda_1 - \lambda_2)P_{it} + \sum_{r=1}^{2} \lambda_{r} P_{it}^{r}$$

**Modèle global**

Notons $D_t^e$ l'indicatrice valant 1 si le mois $t$ se trouve en position $e$ dans le cycle électoral. Le cycle électoral débute le mois du premier tour des élections présidentielles et dure $E$ mois, avec $E=84$ jusqu'en 2002 et $E=60$ ensuite. Nous cherchons donc à estimer :
$$\hat{Y}_{it} = \alpha + \sum_{e=-5}^{0} D_t^e \Bigr[\delta_t^e \bigr[(1 - n\alpha - \theta)(\beta {AN}_{it} + \gamma P_{it}) + \theta G_{it}\bigr] + (1 - \delta_t^e) S_{it}\Bigr] + \sum_{e=1}^{2} D_t^e \Bigr[(1 - n\alpha - \lambda_1 - \lambda_2)P_{it}^{p_1} + \sum_{r=1}^{2} \lambda_{r} P_{it}^{r} \Bigr] + \sum_{e=3}^{E-6} D_t^e \Bigr[(1 - n\alpha - \theta)(\beta {AN}_{it} + \gamma P_{it}) + \theta G_{it} \Bigr]$$

Ce modèle n'est pas linéaire pour plusieurs de ses paramètres. Il est donc impossible de l'estimer avec les MCO, qui de surcroît ne permettraient pas d'assurer le respect des contraintes de compositionnalité. Il faut plutôt procéder à une optimisation sous contrainte, minimisant l'erreur quadratique moyenne ($= \sum_{t,i} (Y_{it} - \hat{Y}_{it})^2$) en imposant que tous les coefficients soient compris entre 0 et 1, et que $\beta + \gamma = 1$.

Il est facile de vérifier qu'en ces circonstances, les contraintes de compositionnalité sont respectées, c'est-à-dire que $\forall t, \sum_{i} \hat{Y}_{it} =1$ : il suffit d'utiliser $\forall t, \sum_{i} {AN}_{it} = \sum_{i} P_{it} = \sum_{i} G_{it} = \sum_{i} S_{it} = 1 $.

Cependant, à la différence des MCO :
- Il n'est pas acquis qu'une solution existe et soit unique.
- Les résidus ne seront pas nécessairement centrés, de sorte que les prédictions pourront conserver un biais systématiques.
- Les approximations asymptotiques ne sont pas valables, ce qui impose de recourir au bootstrapping pour évaluer la significativité des coefficients.

In [ ]:
def compute_y_pred(
    df,
    n, alpha, beta, gamma, theta,
    delta_pre_5, delta_pre_4, delta_pre_3, delta_pre_2, delta_pre_1, delta_pre_0,
    lambda_1, lambda_2):
    
    baseline = (
        beta * (1 - n * alpha - theta) * df["na_share"] +
        gamma * (1 - n * alpha - theta) * df["pres_votes_share"] +
        theta * df["government"])

    y_pred = (
        alpha +
        df["inter_dum"] * baseline +
        df["pre_5"] * (delta_pre_5 * baseline + (1 - delta_pre_5) * df["pres_poll_result"]) +
        df["pre_4"] * (delta_pre_4 * baseline + (1 - delta_pre_4) * df["pres_poll_result"]) +
        df["pre_3"] * (delta_pre_3 * baseline + (1 - delta_pre_3) * df["pres_poll_result"]) +
        df["pre_2"] * (delta_pre_2 * baseline + (1 - delta_pre_2) * df["pres_poll_result"]) +
        df["pre_1"] * (delta_pre_1 * baseline + (1 - delta_pre_1) * df["pres_poll_result"]) +
        df["pre_0"] * (delta_pre_0 * baseline + (1 - delta_pre_0) * df["pres_poll_result"]) +
        df["post_dum"] * (
            (1 - n * alpha - lambda_1 - lambda_2) * df["pres_votes_share"] +
            lambda_1 * df["r2_rank_1"] +
            lambda_2 * df["r2_rank_2"]))

    return y_pred

In [ ]:
def objective(params, data_opt, y_opt):
    alpha = params[0]
    beta = params[1]
    gamma = 1 - beta
    theta = params[2]
    delta_pre_5, delta_pre_4, delta_pre_3, delta_pre_2, delta_pre_1, delta_pre_0 = params[3:9]
    lambda_1, lambda_2 = params[9:11]

    y_pred = compute_y_pred(
        data_opt,
        n, alpha, beta, gamma, theta,
        delta_pre_5, delta_pre_4, delta_pre_3, delta_pre_2, delta_pre_1, delta_pre_0,
        lambda_1, lambda_2)

    return root_mse(y_opt, y_pred)

# Initial guess for the coefficients: alpha, beta, theta, delta_pre_5 to delta_pre_0, lambda_1 and lambda_2
initial_guess = [0.01, 0.7, 0.2, 0.9, 0.85, 0.8, 0.7, 0.5, 0.2, 0.15, 0.1]

# Bounds: all coefficients between 0 and 1
bounds = [(0, 1)] * 11

# Constraints: increasing poll weights during electoral campaigns
constraints = [
    {"type": "ineq", "fun": lambda x: x[3] - x[4]},  # delta_pre_5 > delta_pre_4
    {"type": "ineq", "fun": lambda x: x[4] - x[5]},  # delta_pre_4 > delta_pre_3
    {"type": "ineq", "fun": lambda x: x[5] - x[6]},  # delta_pre_3 > delta_pre_2
    {"type": "ineq", "fun": lambda x: x[6] - x[7]},  # delta_pre_2 > delta_pre_1
    {"type": "ineq", "fun": lambda x: x[7] - x[8]},  # delta_pre_1 > delta_pre_0
]

# Number of political nuances
n = len(model_data_no_journal['political_alignment'].unique())

### Sur l'ensemble des données

In [ ]:
data_opt = model_data_no_journal[
    ["inter_dum", "pre_5", "pre_4", "pre_3", "pre_2", "pre_1", "pre_0", "post_dum", 
     "na_share", "pres_votes_share", "government", "pres_poll_result", "r2_rank_1", "r2_rank_2"]]

y_opt = model_data_no_journal[outcome]

# Set constraints=constraints to use the conditions on delta_pre_x
result = minimize(objective, initial_guess, args=(data_opt, y_opt), bounds=bounds, constraints=[], method='SLSQP')

if result.success:
    alpha, beta, theta, delta_pre_5, delta_pre_4, delta_pre_3, delta_pre_2, delta_pre_1, delta_pre_0, lambda_1, lambda_2 = result.x
    gamma = 1 - beta

    print(f"Optimal alpha: {alpha:.5f}")
    print(f"Optimal beta: {beta:.5f}")
    print(f"Optimal gamma: {gamma:.5f}")
    print(f"Optimal theta: {theta:.5f}")
    print(f"Optimal delta_pre_5: {delta_pre_5:.5f}")
    print(f"Optimal delta_pre_4: {delta_pre_4:.5f}")
    print(f"Optimal delta_pre_3: {delta_pre_3:.5f}")
    print(f"Optimal delta_pre_2: {delta_pre_2:.5f}")
    print(f"Optimal delta_pre_1: {delta_pre_1:.5f}")
    print(f"Optimal delta_pre_0: {delta_pre_0:.5f}")
    print(f"Optimal lambda_1: {lambda_1:.5f}")
    print(f"Optimal lambda_2: {lambda_2:.5f}")
else:
    print("Optimization failed:", result.message)

Lecture
- Période inter-électorale
    - $\theta=0,09$ signifie que 9 % des citations vont aux nuances politiques représentées au gouvernement. Les 91 % restants sont répartis entre l'ensemble des nuances politiques en fonction de leur importance électorale et institutionnelle.
    - $\beta=0,76$ signifie que l'importance électorale et institutionnelle est déterminée à 76 % par la proportion de députés et à 24 % par le score au premier tour de l'élection présidentielle, c'est-à-dire que l'importance institutionnelle prime nettement sur l'importance électorale.
- Période pré-électorale
    - $\delta^e=1$ signifie une absence totale de prise en compte des sondages, le taux de citation est le même qu'en période inter-électorale.
    - $\delta^e=0$ signifie une absence totale de prise en compte de l'importance institutionnelle et électorale du cycle qui s'achève, le taux de citations dépend seulement des sondages.
    - La décroissance des valeurs de $\delta$ témoigne donc d'un poids croissant des sondages durant la campagne électorale (de novembre pour `delta_pre_5` à avril pour `delta_pre_0`).
- Période post-électorale
    - $\lambda_1=0,21$ => la nuance arrivée en tête au premier tour de l'élection présidentielle reçoit 21 % des citations.
    - $\lambda_2=0,07$ => la nuance arrivée en second reçoit 7 % des citations.
    - L'ensemble des nuances se partagent les 72 % de citations restantes en fonction de leur score au premier tour.

Les résultats sont cohérents, sauf l'un qui est inattendu : les sondages ont plus d'importance en novembre qu'en décembre et janvier.

In [ ]:
y_pred = compute_y_pred(
    data_opt,
    n, alpha, beta, gamma, theta,
    delta_pre_5, delta_pre_4, delta_pre_3, delta_pre_2, delta_pre_1, delta_pre_0,
    lambda_1, lambda_2)

r2 = r2_score(y_opt, y_pred)
mr = (y_opt - y_pred).mean()
rmspe = root_mse(y_opt, y_pred)

print(f"R2: {100*r2:.3f}%")
print(f"MR: {100*mr:.5f}%")
print(f"RMSPE: {rmspe:.5f}")

Les résidus moyens sont négatifs. Cela signifie que les prédictions souffrent d'un biais systématique : en moyenne, elles sont inférieures de 0,00018 points de pourcentage aux valeurs observées. Le $R^2$ n'a théoriquement pas de signification lorsque les résidus ne sont pas centrés. L'écart est cependant très faible. Il s'explique probablement par les approximations résultant du passage aux proportions pour les régresseurs, faisant qu'ils ne somment pas toujours exactement à 1. Autorisons-nous à négliger ce problème...

In [ ]:
# Bootstrapping (on months instead of observations to respect compositionnality)
n_bootstraps = 5000
n_blocks = len(model_data_no_journal) // 6

def run_bootstrap_iteration(seed, data):
    np.random.seed(seed)
    sampled_block_ids = np.random.choice(n_blocks, size=n_blocks, replace=True)
    sampled_row_indices = np.concatenate([
        np.arange(block_id * 6, block_id * 6 + 6) for block_id in sampled_block_ids
    ])
    boot_data = data.iloc[sampled_row_indices].reset_index(drop=True)
    boot_data = boot_data[[
        outcome,
        "political_alignment", "inter_dum", "pre_5", "pre_4", "pre_3", "pre_2", "pre_1", "pre_0", "post_dum",
        "na_share", "pres_votes_share", "government", "pres_poll_result", "r2_rank_1", "r2_rank_2"]]
    boot_y = boot_data[outcome]

    result = minimize(objective, initial_guess, args=(boot_data, boot_y), bounds=bounds, constraints=[], method='SLSQP')
    
    if result.success:
        alpha_bst, beta_bst, theta_bst, delta_pre_5_bst, delta_pre_4_bst, delta_pre_3_bst, delta_pre_2_bst, delta_pre_1_bst, delta_pre_0_bst, lambda_1_bst, lambda_2_bst = result.x
        gamma_bst = 1 - beta_bst
        return {
            "alpha": alpha_bst,
            "beta": beta_bst,
            "gamma": gamma_bst,
            "theta": theta_bst,
            "delta_pre_5": delta_pre_5_bst,
            "delta_pre_4": delta_pre_4_bst,
            "delta_pre_3": delta_pre_3_bst,
            "delta_pre_2": delta_pre_2_bst,
            "delta_pre_1": delta_pre_1_bst,
            "delta_pre_0": delta_pre_0_bst,
            "lambda_1": lambda_1_bst,
            "lambda_2": lambda_2_bst
        }
    else:
        return None

results = Parallel(n_jobs=-1)(
    delayed(run_bootstrap_iteration)(seed, model_data_no_journal)
    for seed in tqdm(range(n_bootstraps), desc="Bootstrapping")
)

# Filter out failed runs (None)
bootstrap_results = [r for r in results if r is not None]

In [ ]:
# Using a one-tail approach for pvalues as coefficients are bounded by zero
bootstrap_df = pd.DataFrame(bootstrap_results)
pval = (bootstrap_df <= 0).sum() / len(bootstrap_df)
original_coeffs = pd.Series({
    "alpha": alpha,
    "beta": beta,
    "gamma": gamma,
    "theta": theta,
    "delta_pre_5": delta_pre_5,
    "delta_pre_4": delta_pre_4,
    "delta_pre_3": delta_pre_3,
    "delta_pre_2": delta_pre_2,
    "delta_pre_1": delta_pre_1,
    "delta_pre_0": delta_pre_0,
    "lambda_1": lambda_1,
    "lambda_2": lambda_2})

bootstrap_df = bootstrap_df.describe(percentiles=[0.025, 0.5, 0.975]).T
bootstrap_df["pval"] = pval
bootstrap_df["coeff"] = original_coeffs
bootstrap_df = bootstrap_df.drop(columns = ['count', 'min', '50%', 'max'])
cols = ["coeff"] + [col for col in bootstrap_df.columns if col != "coeff"]
bootstrap_df = bootstrap_df[cols]
bootstrap_df.style

En bootstrappant le modèle, il faut veiller à conserver la compositionnalité des données. Cela signifie que le ré-échantillonnage ne doit pas porter sur des observations isolées, mais plutôt sur des mois, c'est-à-dire des blocs comportant des observations pour l'ensemble des nuances politiques. Après 5000 itérations, tous les coefficients s'avèrent significatifs à plus de 0,001 %, sauf la constante (avec une $p$-value de 45,2 %) et le coefficient du parti arrivé second au premier tour de l'élection présidentielle (avec une $p$-value de 11,5 %).

### Pour la droite et la gauche dans *Le Monde* avant juin 2012

In [ ]:
# Keeping n=6 for consistent results
cutoff = pd.Period('2012-06', freq='M')

data_opt = model_data[
    (model_data["month"] <= cutoff) & 
    (model_data["journal"] == "Le Monde") &
    (model_data["political_alignment"].isin(['Right', 'Left']))
    ][["inter_dum", "pre_5", "pre_4", "pre_3", "pre_2", "pre_1", "pre_0", "post_dum", 
       "na_share", "pres_votes_share", "government", "pres_poll_result", "r2_rank_1", "r2_rank_2"]]

y_opt = model_data[
    (model_data["month"] <= cutoff) & 
    (model_data["journal"] == "Le Monde") &
    (model_data["political_alignment"].isin(['Right', 'Left']))
    ][outcome]

# Set constraints=constraints to use the conditions on delta_pre_x
result = minimize(objective, initial_guess, args=(data_opt, y_opt), bounds=bounds, constraints=[], method='SLSQP')

if result.success:
    alpha_loc, beta_loc, theta_loc, delta_pre_5_loc, delta_pre_4_loc, delta_pre_3_loc, delta_pre_2_loc, delta_pre_1_loc, delta_pre_0_loc, lambda_1_loc, lambda_2_loc = result.x
    gamma_loc = 1 - beta_loc

    print(f"Optimal alpha: {alpha_loc:.5f}")
    print(f"Optimal beta: {beta_loc:.5f}")
    print(f"Optimal gamma: {gamma_loc:.5f}")
    print(f"Optimal theta: {theta_loc:.5f}")
    print(f"Optimal delta_pre_5: {delta_pre_5_loc:.5f}")
    print(f"Optimal delta_pre_4: {delta_pre_4_loc:.5f}")
    print(f"Optimal delta_pre_3: {delta_pre_3_loc:.5f}")
    print(f"Optimal delta_pre_2: {delta_pre_2_loc:.5f}")
    print(f"Optimal delta_pre_1: {delta_pre_1_loc:.5f}")
    print(f"Optimal delta_pre_0: {delta_pre_0_loc:.5f}")
    print(f"Optimal lambda_1: {lambda_1_loc:.5f}")
    print(f"Optimal lambda_2: {lambda_2_loc:.5f}")
else:
    print("Optimization failed:", result.message)

Que se passe-t-il en restreignant l'estimation au journal *Le Monde*, à la période s'achevant en juin 2012, et aux nuances dominantes que sont la droite et la gauche ?
- Concernant les périodes inter-électorales, la représentation à l'Assemblée nationale pèse désormais moins relativement au  score au premier tour des élections présidentielles, avec un coefficient $\beta$ passant de 0,76 à 0,7. La proportion des citations accordée au gouvernement diminue, avec un coefficient $\theta$ passant de 0,09 à 0,81.
- Concernant les périodes électorales, l'importance accordée aux sondages s'avère moindre. Les résultats intriguant obtenus précédemment persiste : les sondages prennent une importance en novembre qui n'est retrouvée qu'en février.
- Concernant les périodes post-électorales, les nuances arrivées en tête au premier tour des élections présidentielles bénéficient de citations plus nombreuses, $\lambda_1$ passant de 0,209 à 0,342 et $\lambda_2$ de 0,068 à 0,268.

In [ ]:
# Performance on restricted dataset
y_pred = compute_y_pred(
    data_opt,
    n, alpha_loc, beta_loc, gamma_loc, theta_loc,
    delta_pre_5_loc, delta_pre_4_loc, delta_pre_3_loc, delta_pre_2_loc, delta_pre_1_loc, delta_pre_0_loc,
    lambda_1_loc, lambda_2_loc)

r2 = r2_score(y_opt, y_pred)
mr = (y_opt - y_pred).mean()
rmspe = root_mse(y_opt, y_pred)

print(f"R2: {100*r2:.3f}%")
print(f"MR: {100*mr:.5f}%")
print(f"RMSPE: {rmspe:.5f}")

Evalué sur les mêmes données, ce modèle s'avère moins performant que le précédent, avec une erreur quadratique augmentant de 39 % et des résidus moyens multipliés par environ 17 000, témoignant de prédictions désormais systématiquement biaisées.

*NB : Les $R^2$ ne sont pas comparables car les échantillons ne sont pas identiques.*

In [ ]:
# Performance on full dataset
y_pred = compute_y_pred(
    model_data_no_journal,
    n, alpha_loc, beta_loc, gamma_loc, theta_loc,
    delta_pre_5_loc, delta_pre_4_loc, delta_pre_3_loc, delta_pre_2_loc, delta_pre_1_loc, delta_pre_0_loc,
    lambda_1_loc, lambda_2_loc)

y_opt = model_data_no_journal[outcome]

r2 = r2_score(y_opt, y_pred)
mr = (y_opt - y_pred).mean()
rmspe = root_mse(y_opt, y_pred)

print(f"R2: {100*r2:.3f}%")
print(f"MR: {100*mr:.5f}%")
print(f"RMSPE: {rmspe:.5f}")

Sur l'échantillon complet, en revanche, ce modèle obtient des performances similaires au précédent, avec des résidus de nouveau centrés et un $R^2$ (désormais comparable) à peine plus faible. Les résidus moyens diminuent et l'erreur quadratique moyenne augmente, quoique de manière très modérée, ce qui témoigne de performances améliorées sur les valeurs usuelles mais dégradée sur les valeurs extrêmes.


Nous verrons plus loin que la répartition des citations n'évolue pas selon les journaux et les périodes. Les résultats de l'estimation sur un échantillon restreint à un journal, deux périodes et deux nuances politiques signifie en fait que ces nuances sont les plus contributives au modèle, car elles représentent les partis mainstream recevant la majorité des citations.

In [ ]:
data = model_data[
    (model_data["month"] <= cutoff) & 
    (model_data["journal"] == "Le Monde") &
    (model_data["political_alignment"].isin(['Right', 'Left']))
    ][[outcome, "political_alignment", 
       "inter_dum", "pre_5", "pre_4", "pre_3", "pre_2", "pre_1", "pre_0", "post_dum", 
       "na_share", "pres_votes_share", "government", "pres_poll_result", "r2_rank_1", "r2_rank_2"]]

n_bootstraps = 1000
n_blocks = len(data) // 6

results = Parallel(n_jobs=-1)(
    delayed(run_bootstrap_iteration)(seed, data)
    for seed in tqdm(range(n_bootstraps), desc="Bootstrapping")
)

# Filter out failed runs
bootstrap_results = [r for r in results if r is not None]

In [ ]:
# Using a one-tail approach for pvalues as coefficients are bounded by zero
bootstrap_df = pd.DataFrame(bootstrap_results)
pval = (bootstrap_df <= 0).sum() / len(bootstrap_df)
original_coeffs = pd.Series({
    "alpha": alpha_loc,
    "beta": beta_loc,
    "gamma": gamma_loc,
    "theta": theta_loc,
    "delta_pre_5": delta_pre_5_loc,
    "delta_pre_4": delta_pre_4_loc,
    "delta_pre_3": delta_pre_3_loc,
    "delta_pre_2": delta_pre_2_loc,
    "delta_pre_1": delta_pre_1_loc,
    "delta_pre_0": delta_pre_0_loc,
    "lambda_1": lambda_1_loc,
    "lambda_2": lambda_2_loc})

bootstrap_df = bootstrap_df.describe(percentiles=[0.025, 0.5, 0.975]).T
bootstrap_df["pval"] = pval
bootstrap_df["coeff"] = original_coeffs
bootstrap_df = bootstrap_df.drop(columns = ['count', 'min', '50%', 'max'])
cols = ["coeff"] + [col for col in bootstrap_df.columns if col != "coeff"]
bootstrap_df = bootstrap_df[cols]
bootstrap_df.style

# 2. Écarts à la norme de représentativité selon les nuances politiques

In [ ]:
model_data_no_journal['y_norm'] = compute_y_pred(
    model_data_no_journal,
    n, alpha, beta, gamma, theta,
    delta_pre_5, delta_pre_4, delta_pre_3, delta_pre_2, delta_pre_1, delta_pre_0,
    lambda_1, lambda_2)
plot_data = model_data_no_journal.copy()
plot_data['abs_residuals'] = 100 * (plot_data[outcome] - plot_data['y_norm'])
plot_data['month'] = plot_data['month'].dt.to_timestamp()

In [ ]:
alignment_groups = [
    (['Right', 'Left', 'Center'],
     {'Right': 'cornflowerblue',
      'Left': 'orchid',
      'Center': 'goldenrod'}),
    (['Far right'],
     {'Far right': 'royalblue'}),
    (['Far left'],
     {'Far left': 'crimson'}),
    (['Other'],
     {'Other': 'forestgreen'})]

fig, axes = plt.subplots(
    4, 1, figsize=(24, 12), sharex=True,
    gridspec_kw={'height_ratios': [3, 1, 1, 1]}
)

for ax, (political_alignments, colors) in zip(axes, alignment_groups):
    alignment_handles = []

    for alignment in political_alignments:
        subset_data = plot_data[plot_data['political_alignment'] == alignment].copy()
        subset_data['MA'] = subset_data[outcome].rolling(window=4).mean()
        
        ax.plot(subset_data['month'], subset_data[outcome], label=None,
                alpha=0.3, color=colors[alignment], linestyle='-')
        ax.plot(subset_data['month'], subset_data['MA'], label=None,
                alpha=0.65, color=colors[alignment], linestyle='dashdot')
        ax.plot(subset_data['month'], subset_data['y_norm'], label=None,
                alpha=1, color=colors[alignment], linestyle='dotted')
        
        alignment_handles.append(Line2D([0], [0], color=colors[alignment], lw=2, label=alignment))

    alignment_legend = ax.legend(handles=alignment_handles, title="Political alignment", loc="upper left")
    ax.add_artist(alignment_legend)

    line_type_handles = [
        Line2D([0], [0], color='black', lw=2, linestyle='-', label="Monthly average"),
        Line2D([0], [0], color='black', lw=2, linestyle='dashdot', label="6 months moving average"),
        Line2D([0], [0], color='black', lw=2, linestyle='dotted', label="Predictions")]
    ax.legend(handles=line_type_handles, title="Values", loc="upper right")

axes[-1].set_xlabel("")

add_shaded_periods(axes, main_elec_months, color='black', alpha=0.1)

plt.suptitle("Quote Distribution by Political Affiliation\nObserved vs. Predicted Values")
plt.tight_layout()
plt.savefig("output/nuances_val_graph.png", dpi=300, bbox_inches='tight')
plt.show()

Concernant la **droite radicale**, le modèle fonctionne plutôt bien. Les résidus augmentent à certaines périodes, qui se rattachent toutefois à des [évènements historiques](https://fr.wikipedia.org/wiki/Chronologie_du_Rassemblement_national) indépendants du modèle :
- Les années 1984-85 correspondent aux premiers succès électoraux du FN, d'abord à des scrutins locaux, qui culminent avec son entrée à l'Assemblée nationale en 1986.
- Les années 1992-93 sont marquées par de nouveaux succès électoraux (aux régionales en 1992) et médiatiques (avec une prestation remarquée de Jean-Marie Le Pen dans *L'Heure de vérité* en 1993).
- L'année 2015 se distingue par les bons scores du FN aux élections départementales et régionales. 

C'est seulement après les élections législatives de 2002, à l'issue desquelles le RN obtient 89 députés, que la droite radicale commence à recevoir systématiquement moins de citations que ne le prévoit le modèle. 

Concernant la **gauche radicale**, les résultats sont très différents : son taux de citation stagne à un niveau faible, largement décorrelé de son importance électorale et institutionnelle, pendant presque toute la période. La participation du PCF au gouvernement en 1981-84 et 1997-2002, par exemple, est à peine visible. C'est seulement à partir de 2016, avec l'émergence de la France Insoumise et la quadripartition concomitante de la vie politique, que la couverture médiatique de la gauche radicale cesse d'être résiduelle. L'écart aux prévisions du modèle se réduit nettement lors des périodes électorales. Il demeure substantiel le reste du temps, mais pas notablement plus élevé que pour la droite radicale.

Concernant la **droite** et la **gauche**, le modèle s'avère globalement bien ajusté. Les périodes où il semble le plus biaisé semblent suivre les dissolutions : entre 1986 et 1988, puis entre 1997 et 2002, et à partir de 2024. Une explication serait que ces dissolutions ont toutes été perdues, de sorte que le score au premier tour de la dernière élection présidentielle ne reflète plus fidèlement l'importance électorale des partis. Les journalistes réduiraient alors l'attention accordée aux perdants des dissolutions, au profits de leurs gagnants. Cette explication est toutefois partielle, car la proportion de députés pèse environ 4 fois plus dans le modèle ($\beta≈0,8$ contre $\gamma≈0,2$) et ne paraît pas susceptible d'expliquer des écarts aussi importants que ceux observés, dépassant 15 points de pourcentage entre les valeurs observées et prédites. Par ailleurs, on observe souvent des tendances qui semblent anticiper les résultats du prochain cycle électoral bien avant le début des campagnes, et la prise en compte des sondages dans le modèle. Ce phénomène est particulièrement visible en 1981-87, 1990-93, 2005-07 et 2011-12.

Concernant finalement le **centre**, qui accède au pouvoir en 2017, il ne paraît pas correctement représenté : le modèle prédit trop de citations 2017-20, et plus assez en 2022-25. Une interprétation serait que les journalistes sont incertains face à un changement fondamental dans la vie politique française, passant d'une bi- à une tri- ou quadri-partition. Ils continueraient ainsi, au moins jusqu'en 2022, à donner la primauté aux nuances dominantes d'hier. Une autre interprétation serait que le gouvernement au centre s'accompagne des négociations permanentes avec la gauche et la droite, amplifiant leur importance institutionnelle. Mais cela serait surtout vrai après 2022, après la perte de sa quasi-majorité absolue à l'Assemblée par le centre.

In [ ]:
alignment_groups = [
    (['Right', 'Left', 'Center'],
     {'Right': 'cornflowerblue',
      'Left': 'orchid',
      'Center': 'goldenrod'}),
    (['Far left', 'Far right', 'Other'],
     {'Far left': 'crimson',
      'Far right': 'royalblue',
      'Other': 'forestgreen'})]

fig, axes = plt.subplots(2, 1, figsize=(24, 8), sharex=True)

for ax, (political_alignments, colors) in zip(axes, alignment_groups):
    alignment_handles = []

    for alignment in political_alignments:
        subset_data = plot_data[plot_data['political_alignment'] == alignment].copy()
        subset_data['MA'] = subset_data['abs_residuals'].rolling(window=6).mean()
        
        ax.plot(subset_data['month'], subset_data['abs_residuals'], label=None,
                alpha=0.2, color=colors[alignment], linestyle='-')
        ax.plot(subset_data['month'], subset_data['MA'], label=None,
                alpha=0.65, color=colors[alignment], linestyle='-.')
        ax.plot(subset_data['month'], [0] * len(subset_data), label=None,
                alpha=1, color='darkgray', linestyle=':')
        
        alignment_handles.append(Line2D([0], [0], color=colors[alignment], lw=2, label=alignment))

    alignment_legend = ax.legend(handles=alignment_handles, title="Political alignment", loc="upper left")
    ax.add_artist(alignment_legend)

    line_type_handles = [
        Line2D([0], [0], color='black', lw=2, linestyle='-', label="Monthly average"),
        Line2D([0], [0], color='black', lw=2, linestyle='-.', label="6 months moving average"),
        Line2D([0], [0], color='black', lw=2, linestyle=':', label="Predictions")
    ]
    ax.legend(handles=line_type_handles, title="Values", loc="upper right")

axes[-1].set_xlabel("")

add_shaded_periods(axes, main_elec_months, color='black', alpha=0.1)

plt.suptitle("Quote Distribution by Political Affiliation\nAbsolute Residuals (%)")
plt.tight_layout()
plt.savefig("output/nuances_res_graph.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
def compute_proportional_metrics(y_norm, y_true):
    y_norm = np.array(y_norm)
    y_true = np.array(y_true)
    
    # Initialize arrays
    TP = np.zeros_like(y_norm)
    TN = np.zeros_like(y_norm)
    FP = np.zeros_like(y_norm)
    FN = np.zeros_like(y_norm)
    
    # Calculate difference
    diff = y_norm - y_true
    
    # Case 1: y_norm - y_true = 0 (perfect prediction)
    perfect_mask = (diff == 0)
    TP[perfect_mask] = 1
    FP[perfect_mask] = 0
    FN[perfect_mask] = 0
    
    # Case 2: y_norm - y_true > 0 (over-prediction)
    over_mask = (diff > 0)
    TP[over_mask] = y_true[over_mask] / y_norm[over_mask]
    FP[over_mask] = diff[over_mask] / y_norm[over_mask]
    FN[over_mask] = 0
    
    # Case 3: y_norm - y_true < 0 (under-prediction)
    under_mask = (diff < 0)
    TP[under_mask] = 1
    FN[under_mask] = - diff[under_mask] / y_true[under_mask]
    FP[under_mask] = 0
    
    return {
        'TP': TP.mean(),
        'FP': FP.mean(),
        'FN': FN.mean()}

In [ ]:
cutoff1 = pd.Period('2002-06', freq='M')
cutoff2 = pd.Period('2017-06', freq='M')

results = []

for period_label, period_filter in {
    '1981-2002': model_data_no_journal["month"] < cutoff1,
    '2002-2017': (model_data_no_journal["month"] >= cutoff1) & (model_data_no_journal["month"] < cutoff2),
    '2017-2024': model_data_no_journal["month"] >= cutoff2
}.items():
    period_data = model_data_no_journal[period_filter]
    
    for alignment in period_data["political_alignment"].unique():

        subset_data = period_data[period_data["political_alignment"] == alignment]
        y = subset_data[outcome]
        y_norm = subset_data['y_norm']
        mr = (y - y_norm).mean()
        metrics = compute_proportional_metrics(y_norm, y)
        results.append({
            'period': period_label,
            'alignment': alignment,
            'mean residuals': mr,
            '% correct predictions': 100 * metrics['TP'],
            '% excess predictions': 100 * metrics['FP'],
            '% missing predictions': 100 * metrics['FN']})

summary = pd.DataFrame(results)
summary["alignment"] = pd.Categorical(summary["alignment"], categories=nuances_order, ordered=True)
summary_tbl = summary.pivot_table(index="alignment",
                                    columns="period",
                                    values=["% correct predictions",
                                            "% excess predictions",
                                            "% missing predictions"],
                                    observed=False)
summary_tbl = summary_tbl.reset_index()
summary_tbl.columns.names = [None, None]
summary_tbl.style.hide(axis="index").format({col: "{:.3f}" for col in summary_tbl.columns[1:]})

Ce tableau présente les proportions des proportions de citations correctement prédites. Les valeurs s'interprètent comme les différences entre les courbes des valeurs prédites et réelles.
- Correct predictions = proportion des citations prédites correspondant à des citations réelles (lorsque la courbe des citations prédites est <= à celle des citations réelles)
- Excess predictions = proportion des citations prédites excédant les citations réelles (lorsque la courbe des citations prédites est >= à celle des citations réelles)
- Missing predictions = proportion des citations réelles non-prédites par le modèle (lorsque la courbe des citations réelles est >= à celle des citations prédites)

In [ ]:
summary_tbl = summary.pivot_table(index="alignment",
                                    columns="period",
                                    values="mean residuals",
                                    observed=False)
summary_tbl = summary_tbl.reset_index()
summary_tbl.style.hide(axis="index").format({col: "{:.3f}" for col in summary_tbl.columns[1:]})

Ce tableau présente les résidus moyens. Ils sont positifs lorsque les citations réelles sont plus nombreuses que les citations prédites, c'est-à-dire lorsque la couverture médiatique s'avère biaisée favorablement pour la nuance concernée. 

In [ ]:
cutoff1 = pd.Period('2002-06', freq='M').to_timestamp()
cutoff2 = pd.Period('2017-06', freq='M').to_timestamp()

plot_data["period"] = pd.NA  
plot_data.loc[plot_data["month"] < cutoff1, "period"] = "1981-2002"
plot_data.loc[(plot_data["month"] >= cutoff1) & (plot_data["month"] < cutoff2), "period"] = "2002-2017"
plot_data.loc[plot_data["month"] >= cutoff2, "period"] = "2017-2024"

alignment_ref = 'Right'
period_ref = '1981-2002'

model = smf.ols(
    "Q('abs_residuals') ~ C(political_alignment,Treatment(reference='{}')) + C(period, Treatment(reference='{}'))".format(alignment_ref, period_ref),
    data=plot_data
).fit(covtype='HC3')
model.summary()

Pour une analyse plus rigoureuse, on peut régresser les résidus sur nos variables d'intérêt.
- Les coefficients des périodes ne sont pas significatifs, ce qui témoigne d'une absence d'évolution temporelle des écarts à la norme de représentativité.
- Les coefficients des nuances politiques sont très significatifs, ce qui signifie que les écarts à la norme de représentativité ne sont pas neutres politiquement. Cf. *infra* pour l'interprétation des coefficients.

In [ ]:
mr_right = plot_data[plot_data['political_alignment'] == 'Right']['abs_residuals'].mean()
mr_left = plot_data[plot_data['political_alignment'] == 'Left']['abs_residuals'].mean()
diff = mr_left - mr_right

print(f"Mean residuals for the right: {mr_right:.5f}")
print(f"Mean residuals for the left: {mr_left:.5f}")
print(f"Difference between residuals for the left and the right: {diff:.4f}")

Pour `C(political_alignment, Treatment(reference='Right'))[T.Left]`, le coefficient est de -2,6974. Cela signifie que les résidus moyens de la gauche sont inférieurs de 2,7 à ceux de la droite, en moyenne sur l'ensemble des périodes.
- Les résidus moyens de la droite valent 4,852 : en moyenne, la proportion des citations des personnalités de droite est supérieure de 4,852 points de pourcentage à celle prédite par le modèle, ce qui signifie que la droite est avantagée par rapport à la norme de représentativité.
- Les résidus moyens de la gauche valent 2,154 : en moyenne, la proportion des citations des personnalités de gauche est supérieure de 2,154 points de pourcentage à celle prédite par le modèle, ce qui signifie que la gauche est avantagée aussi, mais 2,25 fois moins environ.
- La différence entre ces deux valeurs est de -2,697 : la gauche est désavantagée par rapport à la droite, à hauteur de 2,697 points de pourcentage (même si elles sont avantagées toutes les deux).

In [ ]:
cutoff1 = pd.Period('2002-06', freq='M').to_timestamp()
cutoff2 = pd.Period('2017-06', freq='M').to_timestamp()

plot_data["period"] = pd.NA  
plot_data.loc[plot_data["month"] < cutoff1, "period"] = "1981-2002"
plot_data.loc[(plot_data["month"] >= cutoff1) & (plot_data["month"] < cutoff2), "period"] = "2002-2017"
plot_data.loc[plot_data["month"] >= cutoff2, "period"] = "2017-2024"

alignment_ref = 'Right'
period_ref = '1981-2002'

model = smf.ols(
    "Q('abs_residuals') ~ C(political_alignment,Treatment(reference='{}')) * C(period, Treatment(reference='{}'))".format(alignment_ref, period_ref),
    data=plot_data
).fit(covtype='HC3')
model.summary()

Pour se rapprocher des tableaux précédents, il faut introduire une interaction entre les nuances politiques et les périodes. L'interprétation devient plus complexe, car les coefficients sont expriment désormais la différence des résidus moyens par rapport à une période * nuance de référence, à savoir la droite en 1981-2002.
- L'intercept correspond à la moyenne des résidus pour cette période * nuance.
- Le coefficient d'une période correspond à la différence *pour la droite* avec la période de référence.
- Le coefficient d'une nuance politique correspond à la différence *en 1981-2002* entre cette nuance et la droite.
- Les coefficients des termes d'interaction entre une nuance et une période correspondent à l'évolution de l'écart entre cette nuance et la droite, entre cette période et 1981-2002.

Concernant l'extrême-droite, sa couverture médiatique était moins favorable que celle de la droite en première période. Ce désavantage s'est amplifié en seconde période, mais réduit en troisème.

In [ ]:
mr_right_8102 = plot_data[(plot_data['political_alignment'] == 'Right') & (plot_data['period'] == '1981-2002')]['abs_residuals'].mean()
mr_left_8102 = plot_data[(plot_data['political_alignment'] == 'Left') & (plot_data['period'] == '1981-2002')]['abs_residuals'].mean()
mr_right_1724 = plot_data[(plot_data['political_alignment'] == 'Right') & (plot_data['period'] == '2017-2024')]['abs_residuals'].mean()
mr_left_1724 = plot_data[(plot_data['political_alignment'] == 'Left') & (plot_data['period'] == '2017-2024')]['abs_residuals'].mean()

print("Intercept")
print(f"{mr_right_8102:.4f}")
print("C(period, Treatment(reference='1981-2002'))[T.2017-2024]")
print(f"{(mr_right_1724 - mr_right_8102):.4f}")
print("C(political_alignment, Treatment(reference='Right'))[T.Left]")
print(f"{(mr_left_8102 - mr_right_8102):.4f}")
print("C(political_alignment, Treatment(reference='Right'))[T.Left]:C(period, Treatment(reference='1981-2002'))[T.2017-2024]")
print(f"{(mr_left_1724 - mr_right_1724) - (mr_left_8102 - mr_right_8102):.4f}")

# 2. Écarts à la norme de représentativité selon les journaux

In [ ]:
model_data['y_norm'] = compute_y_pred(
    model_data,
    n, alpha, beta, gamma, theta,
    delta_pre_5, delta_pre_4, delta_pre_3, delta_pre_2, delta_pre_1, delta_pre_0,
    lambda_1, lambda_2)
plot_data = model_data.copy()
plot_data['abs_residuals'] = 100 * (plot_data[outcome] - plot_data['y_norm'])
plot_data['month'] = plot_data['month'].dt.to_timestamp()

colors = {
    'Le Figaro': 'mediumblue',
    'Libération': 'tomato',
    'Le Monde': 'dimgrey',
    'La Croix': 'goldenrod',
    'Médiapart': 'limegreen'}

alignments = [
    "Far right",
    "Right",
    "Center",
    "Left",
    "Far left"]

n_alignments = len(alignments)

In [ ]:
fig, axes = plt.subplots(n_alignments, 1, figsize=(24, 4 * n_alignments), sharex=True)

for i, alignment in enumerate(alignments):
    ax = axes[i]
    subset_data = plot_data[plot_data['political_alignment'] == alignment]

    for journal in subset_data['journal'].unique():
        sub_subset_data = subset_data[subset_data['journal'] == journal].copy()
        sub_subset_data['MA'] = sub_subset_data[outcome].rolling(window=12).mean()
        ax.plot(sub_subset_data['month'], sub_subset_data[outcome], label=journal,
                alpha=0.5, color=colors[journal], linestyle='-')

    ax.plot(subset_data['month'], subset_data['y_norm'], color='black', alpha=0.8, linestyle='dotted')
    ax.set_title(f"{alignment}")
    ax.legend()

add_shaded_periods(axes, main_elec_months, color='black', alpha=0.1)

plt.suptitle("""
Quote Distribution by Political Affiliation and Journal
Observed vs. Predicted Values
""")
plt.tight_layout()
plt.savefig("output/journals_val_graph.png", dpi=300, bbox_inches='tight')
plt.show()

On constate que les journaux suivent globalement la norme de représentativité, avec des courbes globalement très proches. Des différences éditoriales mineures mais prévisibles apparaissent : *Le Figaro* cite davantage des personnalités de droite, Libération des personnalités de *gauche*... Concernant l'extrême droite, aucune journal ne se distingue par un traitement particulier.

In [ ]:
fig, axes = plt.subplots(n_alignments, 1, figsize=(24, 4 * n_alignments), sharex=True)

for i, alignment in enumerate(alignments):
    ax = axes[i]
    subset_data = plot_data[plot_data['political_alignment'] == alignment]

    for journal in subset_data['journal'].unique():
        sub_subset_data = subset_data[subset_data['journal'] == journal].copy()
        sub_subset_data['MA'] = sub_subset_data['abs_residuals'].rolling(window=12).mean()
        ax.plot(sub_subset_data['month'], sub_subset_data['abs_residuals'], label=journal,
                alpha=0.5, color=colors[journal], linestyle='-')

    ax.plot(subset_data['month'], subset_data['y_norm'], color='black', alpha=0.8, linestyle='dotted')
    ax.set_title(f"{alignment}")
    ax.legend()

add_shaded_periods(axes, main_elec_months, color='black', alpha=0.1)

plt.suptitle("""
Quote Distribution by Political Affiliation and Journal\n
Absolute Residuals (%)
""")
plt.tight_layout()
plt.savefig("output/journals_res_graph.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
cutoff1 = pd.Period('2002-06', freq='M')
cutoff2 = pd.Period('2017-06', freq='M')

results = []

for period_label, period_filter in {
    '1981-2002': model_data["month"] < cutoff1,
    '2002-2017': (model_data["month"] >= cutoff1) & (model_data["month"] < cutoff2),
    '2017-2024': model_data["month"] >= cutoff2
}.items():
    period_data = model_data[period_filter]
    
    for alignment in period_data["political_alignment"].unique():
        subset_data = period_data[period_data["political_alignment"] == alignment]
        
        for journal in subset_data["journal"].unique():
            sub_subset_data = subset_data[subset_data["journal"] == journal]
            y = sub_subset_data[outcome]
            y_norm = sub_subset_data['y_norm']
            mr = (y - y_norm).mean()
            metrics = compute_proportional_metrics(y_norm, y)
            results.append({
                'period': period_label,
                'alignment': alignment,
                'journal': journal,
                'mean residuals': mr,
                '% correct predictions': 100 * metrics['TP'],
                '% excess predictions': 100 * metrics['FP'],
                '% missing predictions': 100 * metrics['FN']})

        

summary = pd.DataFrame(results)
summary["alignment"] = pd.Categorical(summary["alignment"], categories=nuances_order, ordered=True)
summary_tbl = summary.pivot_table(index=(["alignment", "journal"]),
                                    columns="period",
                                    values=["% correct predictions",
                                            "% excess predictions",
                                            "% missing predictions"],
                                    observed=False)
summary_tbl = summary_tbl.reset_index()
summary_tbl.columns.names = [None, None]
summary_tbl.style.hide(axis="index").format({col: "{:.3f}" for col in summary_tbl.columns[2:]})

In [ ]:
summary_tbl = summary.pivot_table(index=(["alignment", "journal"]),
                                    columns="period",
                                    values="mean residuals",
                                    observed=False)
summary_tbl = summary_tbl.reset_index()
summary_tbl.style.hide(axis="index").format({col: "{:.3f}" for col in summary_tbl.columns[2:]})

In [ ]:
cutoff1 = pd.Period('2002-06', freq='M').to_timestamp()
cutoff2 = pd.Period('2017-06', freq='M').to_timestamp()

plot_data["period"] = pd.NA  
plot_data.loc[plot_data["month"] < cutoff1, "period"] = "1981-2002"
plot_data.loc[(plot_data["month"] >= cutoff1) & (plot_data["month"] < cutoff2), "period"] = "2002-2017"
plot_data.loc[plot_data["month"] >= cutoff2, "period"] = "2017-2024"

alignment_ref = 'Right'
journal_ref = 'Le Monde'
period_ref = '1981-2002'

model = smf.ols(
    "Q('abs_residuals') ~ C(political_alignment,Treatment(reference='{}')) + C(journal, Treatment(reference='{}')) + C(period, Treatment(reference='{}'))".format(alignment_ref, journal_ref, period_ref),
    data=plot_data
).fit(covtype='HC3')
model.summary()

Comme précédemment, la régression des résidus sur les variables d'intérêt permet des interprétations plus simples (mais surtout grâce à l'absence d'interactions) et plus rigoureuses (grace aux $p$-values). Les différences entre journaux ne sont pas significatives, de même que les différences entre périodes. Les écarts entre les nuances politiques demeurent très significatifs mais se réduisent légèrement (en conservant la même interprétation).

In [ ]:
mr_right = plot_data[plot_data['political_alignment'] == 'Right']['abs_residuals'].mean()
mr_left = plot_data[plot_data['political_alignment'] == 'Left']['abs_residuals'].mean()
diff = mr_left - mr_right

print(f"Mean residuals for the right: {mr_right:.3f}")
print(f"Mean residuals for the left: {mr_left:.3f}")
print(f"Difference between residuals for the left and the right: {diff:.3f}")

In [ ]:
model = smf.ols(
    "Q('abs_residuals') ~ C(political_alignment,Treatment(reference='{}')) * C(journal, Treatment(reference='{}')) + C(period, Treatment(reference='{}'))".format(alignment_ref, journal_ref, period_ref),
    data=plot_data
).fit(covtype='HC3')
model.summary()